# Week 3 Day 3 — AFL Chat Agent (scope, retrieval, memory)

Uses Day-1 tables for **exact** stats. Fact cards add light text search.
Off-topic asks should be refused with an AFL redirect.


## Setup

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

from src.prompts import SYSTEM_PROMPT, REFUSAL_EXAMPLES
from src.fact_cards import build_fact_cards
from src.tools import get_team_h2h_record, get_player_season_stats, get_recent_team_results
from src.agent import chat, reset_session, build_agent
from src.data_access import team_h2h_record, player_season_stats

build_fact_cards()
print(SYSTEM_PROMPT[:400], '...')


## Task 1 — Scope & refusals

In [ ]:
for q, a in REFUSAL_EXAMPLES:
    print('Q:', q)
    print('A:', a)
    print('---')


## Task 2 — Structured lookups (exact)

In [ ]:
print(json.dumps(team_h2h_record('Geelong Cats', 'Richmond Tigers', since_year=2015), indent=2))
print(json.dumps(player_season_stats('Gary Ablett', 2018, team='Geelong Cats'), indent=2)[:800])


Stats use pandas on feature/season tables. Fact cards are only for blurbs —
see `docs/retrieval_design.md`.


## Task 3 — Agent + grounding

In [ ]:
reset_session('nb-demo')
out = chat('What is Geelong Cats head-to-head record versus Richmond Tigers since 2015?', session_id='nb-demo')
print(out['answer'])
print('tools:', len(out['tool_log']))
print('grounding:', {k: out['grounding'][k] for k in ['tool_calls','numbers_in_answer','numbers_missing_from_tools','grounded_ok']})


## Task 4 — Multi-turn memory

In [ ]:
reset_session('nb-memory')
turns = [
    'Give me recent results for Sydney Swans.',
    'Who was the last opponent in that list?',
    'Compare that to Sydney vs Melbourne Demons head-to-head overall.',
    'Pick a well-known Swans era player if you can from fact cards, then we can look up season stats.',
    'What about Gary Ablett 2018 season disposals average for Geelong Cats?',
]
for t in turns:
    print('USER:', t)
    out = chat(t, session_id='nb-memory')
    print('BOT:', out['answer'][:500])
    print('(tools', len(out['tool_log']), ')')
    print('---')


## Task 5 — Guardrails

Run `python evaluate_guardrails.py` for the 15+ prompt suite and report in `results/guardrail_eval.md`.


In [ ]:
from pathlib import Path
p = Path('results/guardrail_eval.md')
print(p.read_text(encoding='utf-8')[:1500] if p.exists() else 'Run evaluate_guardrails.py first')
